In [0]:
# ── Gold: club_league_table ──────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql import Window

games = spark.table("football_catalog.silver.fact_games")
clubs = spark.table("football_catalog.silver.dim_clubs").filter("is_current = True")

print("--- Processing: club_league_table ---")

# Calculate Home Match Stats
home_stats = games.select(
    F.col("competition_id"), 
    F.col("season"), 
    F.col("home_club_id").alias("club_id"),
    F.col("home_club_goals").alias("goals_for"),
    F.col("away_club_goals").alias("goals_against"),
    F.when(F.col("home_club_goals") > F.col("away_club_goals"), 3)
     .when(F.col("home_club_goals") == F.col("away_club_goals"), 1)
     .otherwise(0).alias("points"),
    F.when(F.col("home_club_goals") > F.col("away_club_goals"), 1).otherwise(0).alias("wins"),
    F.when(F.col("home_club_goals") == F.col("away_club_goals"), 1).otherwise(0).alias("draws"),
    F.when(F.col("home_club_goals") < F.col("away_club_goals"), 1).otherwise(0).alias("losses")
)

# Calculate Away Match Stats
away_stats = games.select(
    F.col("competition_id"), 
    F.col("season"), 
    F.col("away_club_id").alias("club_id"),
    F.col("away_club_goals").alias("goals_for"),
    F.col("home_club_goals").alias("goals_against"),
    F.when(F.col("away_club_goals") > F.col("home_club_goals"), 3)
     .when(F.col("away_club_goals") == F.col("home_club_goals"), 1)
     .otherwise(0).alias("points"),
    F.when(F.col("away_club_goals") > F.col("home_club_goals"), 1).otherwise(0).alias("wins"),
    F.when(F.col("away_club_goals") == F.col("home_club_goals"), 1).otherwise(0).alias("draws"),
    F.when(F.col("away_club_goals") < F.col("home_club_goals"), 1).otherwise(0).alias("losses")
)

# Combine Home and Away, then aggregate by club and season
combined_stats = home_stats.unionAll(away_stats)

league_table = (combined_stats
    .groupBy("competition_id", "season", "club_id")
    .agg(
        F.count("*").alias("matches_played"),
        F.sum("wins").alias("wins"),
        F.sum("draws").alias("draws"),
        F.sum("losses").alias("losses"),
        F.sum("goals_for").alias("goals_for"),
        F.sum("goals_against").alias("goals_against"),
        (F.sum("goals_for") - F.sum("goals_against")).alias("goal_difference"),
        F.sum("points").alias("points")
    )
)

# Join with dim_clubs to get the club name
gold_league_table = (league_table
    .join(clubs, "club_id", "left")
    .select(
        "competition_id", "season", "name", "matches_played", 
        "wins", "draws", "losses", "goals_for", "goals_against", 
        "goal_difference", "points"
    )
)

# Rank clubs by points, then goal difference to establish league position
window_spec = Window.partitionBy("competition_id", "season").orderBy(
    F.desc("points"), F.desc("goal_difference"), F.desc("goals_for")
)
gold_league_final = gold_league_table.withColumn("league_position", F.rank().over(window_spec))

(gold_league_final.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("football_catalog.gold.club_league_table"))

print("SUCCESS: club_league_table created")